# Explore regional electricity consumption and choose a model

Exploration notebook: it describes the data, builds the features and tests one model per region on 2024.
The production job lives in `eco2mix_regional_production.ipynb`, which holds only what the job needs.

Data: éCO2mix regional consumption from RTE, published by Open Data Réseaux Énergies (ODRÉ) under the Licence Ouverte 2.0,
read through the view `regional_consumption_hourly`: one row per region and per hour, in UTC, for the 12 regions of mainland France
(Corsica is not part of the dataset).

The model is trained **once per region**: Spark distributes the 12 regions across its executors, which train them side by side.

Kernel: **Python 3 (ipykernel)**.

## 1. Set up

`holidays` provides the French public holidays, `scikit-learn` the regression trained in each region.
The notebook server installs both, from its **Dependencies**.

In [ ]:
from forepaas.dwh import connect
from pyspark.sql import functions as F

source = connect("dwh/default_dataset/")
spark = source.spark
spark.sql(f"USE {source.catalog}.{source.schema}")

# Store and compute in UTC; Paris time is derived explicitly where needed
spark.conf.set("spark.sql.session.timeZone", "UTC")

print(f"Spark {spark.version}, catalog {source.catalog}")

## 2. Load the regional consumption

`regional_consumption_hourly` is written in SQL that both Trino and Spark understand.
Calendar columns follow local life, so they are computed in Paris time here.

In [ ]:
def with_paris_time(df):
    return (
        df.withColumn("hour_paris", F.from_utc_timestamp(F.col("hour_utc").cast("timestamp"), "Europe/Paris"))
        .withColumn("hour_of_day", F.hour("hour_paris"))
        .withColumn("year", F.year("hour_paris"))
        .withColumn("month", F.month("hour_paris"))
    )

regional = with_paris_time(spark.table("regional_consumption_hourly")).cache()

print(f"{regional.count()} region-hours")
regional.groupBy("region_code", "region_name").agg(
    F.count("*").alias("hours"),
    F.count("consumption_mw").alias("measured_hours"),
    F.min("hour_utc").alias("first_hour"),
    F.max("hour_utc").alias("last_hour"),
).orderBy("region_code").show(truncate=False)

12 regions, from 1 January 2013, with the same number of hours each.

## 3. Explore

### 3.1 Coverage, year by year

In [ ]:
coverage = (
    regional.where(F.col("consumption_mw").isNotNull())
    .groupBy("region_name")
    .pivot("year")
    .count()
    .orderBy("region_name")
    .toPandas()
    .set_index("region_name")
)
coverage

A full year shows 8,759 measured hours (8,783 in leap years): the hour that occurs twice in autumn appears only once in the data.
From 2013 to 2025, every year is complete for the 12 regions.

### 3.2 How big is each region?

In [ ]:
import matplotlib.pyplot as plt

measured = regional.where(F.col("consumption_mw").isNotNull())

size = (
    measured.where(F.col("year") == 2024)
    .groupBy("region_name")
    .agg((F.avg("consumption_mw") / 1000).alias("consumption_gw"))
    .orderBy("consumption_gw")
    .toPandas()
)
size.plot(x="region_name", y="consumption_gw", kind="barh", legend=False, figsize=(8, 5))
plt.title("Average consumption in 2024")
plt.xlabel("GW")
plt.ylabel("")
plt.show()

Regions differ by a factor of several: an error of 300 MW is small for Île-de-France and large for Centre-Val de Loire.
Errors are therefore compared in percent (MAPE).

### 3.3 Same daily shape, different heating

In [ ]:
profile = (
    measured.where((F.col("year") == 2024) & (F.weekday("hour_paris") < 5))
    .groupBy("region_name", "hour_of_day")
    .agg(F.avg("consumption_mw").alias("mw"))
    .toPandas()
    .pivot(index="hour_of_day", columns="region_name", values="mw")
)
(profile / profile.mean()).plot(figsize=(11, 5), legend=False, alpha=0.8)
plt.title("Weekday profile by region, relative to its own average (2024)")
plt.xlabel("Hour of day (Paris time)")
plt.ylabel("Ratio to the regional average")
plt.xticks(range(24))
plt.grid(True, alpha=0.3)
plt.show()

All regions follow the same daily cycle, with a night low and two peaks.

In [ ]:
heating = (
    measured.where(F.col("year") == 2024)
    .groupBy("region_name")
    .agg(
        (F.avg(F.when(F.col("month") == 1, F.col("consumption_mw")))
         / F.avg(F.when(F.col("month") == 7, F.col("consumption_mw")))).alias("january_to_july")
    )
    .orderBy("january_to_july")
    .toPandas()
)
heating.plot(x="region_name", y="january_to_july", kind="barh", legend=False, figsize=(8, 5))
plt.title("January consumption relative to July, 2024")
plt.xlabel("Ratio")
plt.ylabel("")
plt.show()

The winter increase differs from one region to another: regions with more electric heating are more sensitive to cold.
This is why one model per region makes sense.

## 4. Build the features

Each hour gets:

| Feature | Why |
|---|---|
| `hour_day` | Hour of day crossed with the type of day (Monday to Sunday, public holidays counted as Sundays): each type of day keeps its own shape |
| `hour_lag_day` | The same for the day 48 hours before: a Tuesday forecast from a Sunday needs to know it |
| `month` | The season |
| `lag_48h` | Consumption of the region at the same hour two days before |
| `lag_168h` | Consumption of the region at the same hour one week before |

**Why 48 hours and not 24.** The job runs in the morning of day D to forecast day D+1: day D is not over, the latest complete day is D-1.
**Lags by time, not by row**: each lag is a join on region and time, so missing hours never shift the data.

In [ ]:
import holidays

holiday_dates = spark.createDataFrame(
    [(day, name) for day, name in sorted(holidays.France(years=range(2012, 2030)).items())],
    ["date_paris", "holiday"],
)
print(f"{holiday_dates.count()} public holidays")


def with_calendar(df):
    """Day types in Paris time, public holidays counted as Sundays (6)."""
    hol = F.broadcast(holiday_dates.select("date_paris", "holiday", F.lit(1).alias("is_holiday")))
    lag_hol = F.broadcast(holiday_dates.select(F.col("date_paris").alias("lag_date"), F.lit(1).alias("lag_is_holiday")))
    lag_time = F.col("hour_paris") - F.expr("INTERVAL 48 HOURS")
    return (
        with_paris_time(df)
        .withColumn("date_paris", F.to_date("hour_paris"))
        .withColumn("lag_date", F.to_date(lag_time))
        .join(hol, "date_paris", "left")
        .join(lag_hol, "lag_date", "left")
        .withColumn("day_type", F.when(F.col("is_holiday") == 1, 6).otherwise(F.weekday("hour_paris")))
        .withColumn("lag_day_type", F.when(F.col("lag_is_holiday") == 1, 6).otherwise(F.weekday(lag_time)))
        .withColumn("hour_day", F.concat_ws("_", F.col("hour_of_day").cast("string"), F.col("day_type").cast("string")))
        .withColumn("hour_lag_day", F.concat_ws("_", F.col("hour_of_day").cast("string"), F.col("lag_day_type").cast("string")))
        .drop("is_holiday", "lag_is_holiday", "lag_date")
    )


def with_lags(df, hourly):
    """Join each region's consumption 48 and 168 hours earlier, on region and time.

    The dataset follows Paris time: on the day clocks go back, one UTC hour is missing.
    A missing hour takes the consumption of the hour before.
    """
    for hours in (48, 168):
        for shift in (0, 1):
            lag = hourly.select(
                "region_code",
                (F.col("hour_utc") + F.expr(f"INTERVAL {hours + shift} HOURS")).alias("hour_utc"),
                F.col("consumption_mw").alias(f"lag_{hours}h_{shift}"),
            )
            df = df.join(lag, ["region_code", "hour_utc"], "left")
        df = df.withColumn(
            f"lag_{hours}h", F.coalesce(f"lag_{hours}h_0", f"lag_{hours}h_1")
        ).drop(f"lag_{hours}h_0", f"lag_{hours}h_1")
    return df


hourly = spark.table("regional_consumption_hourly")
dataset = (
    with_lags(with_calendar(hourly), hourly)
    .where(F.col("consumption_mw").isNotNull())
    .dropna(subset=["lag_48h", "lag_168h"])
    .cache()
)
print(f"{dataset.count()} region-hours with both lags")

In [ ]:
(
    dataset.where(
        (F.col("region_code") == 11)
        & (F.col("hour_of_day") == 8)
        & F.col("hour_paris").between("2024-12-23 00:00:00", "2024-12-28 00:00:00")
    )
    .select("region_name", "hour_paris", "holiday", "day_type", "hour_day", "hour_lag_day")
    .orderBy("hour_paris")
    .show(truncate=False)
)

Île-de-France at 8 a.m., around Christmas 2024 (23 December is a Monday).
On 25 December, `day_type` is 6: the holiday is counted as a Sunday.
On 27 December, the day 48 hours before is Christmas, so `hour_lag_day` also ends with 6.

## 5. One model per region, in parallel

`groupBy("region_code").applyInPandas(...)` sends the rows of each region to one Python worker,
which trains a scikit-learn regression and forecasts the target rows.
Regions run side by side, as many at a time as there are executor cores: in a Custom PySpark action, this is set by the number and size of instances in the action's resources.

The same function serves the evaluation here and the production job in section 7: rows marked `train` fit the model,
rows marked `target` receive a forecast.

In [ ]:
FEATURE_CATEGORIES = ["hour_day", "hour_lag_day", "month_name"]
FEATURE_NUMBERS = ["lag_48h", "lag_168h"]
MODEL_COLUMNS = ["region_code", "hour_utc", "split", "consumption_mw"] + FEATURE_CATEGORIES + FEATURE_NUMBERS
PREDICTION_SCHEMA = "region_code long, hour_utc timestamp_ntz, predicted_mw double, train_hours long"


def fit_and_forecast(rows):
    """Runs once per region, on a pandas DataFrame holding all the rows of that region."""
    import pandas as pd
    from sklearn.compose import ColumnTransformer
    from sklearn.linear_model import LinearRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder

    train = rows[rows["split"] == "train"]
    target = rows[rows["split"] == "target"]
    if train.empty or target.empty:
        return pd.DataFrame(columns=["region_code", "hour_utc", "predicted_mw", "train_hours"])

    model = Pipeline([
        ("encode", ColumnTransformer(
            [("calendar", OneHotEncoder(handle_unknown="ignore"), FEATURE_CATEGORIES)],
            remainder="passthrough",
        )),
        ("regress", LinearRegression()),
    ])
    model.fit(train[FEATURE_CATEGORIES + FEATURE_NUMBERS], train["consumption_mw"])

    forecast = target[["region_code", "hour_utc"]].copy()
    forecast["predicted_mw"] = model.predict(target[FEATURE_CATEGORIES + FEATURE_NUMBERS])
    forecast["train_hours"] = len(train)
    return forecast


def prepare(df, split):
    return df.withColumn("split", F.lit(split)).withColumn("month_name", F.col("month").cast("string")).select(MODEL_COLUMNS)

### Train on 2022-2023, test on 2024

In [ ]:
evaluation_rows = prepare(dataset.where(F.col("year").between(2022, 2023)), "train").unionByName(
    prepare(dataset.where(F.col("year") == 2024), "target")
)

predictions = (
    evaluation_rows.groupBy("region_code")
    .applyInPandas(fit_and_forecast, schema=PREDICTION_SCHEMA)
    .join(dataset.select("region_code", "region_name", "hour_utc", "hour_paris", "consumption_mw", "lag_168h"),
          ["region_code", "hour_utc"])
    .cache()
)
print(f"{predictions.count()} forecasts for 2024")

Notebook sessions do not appear in the Spark History Server, which only lists the runs of actions and workflows.
The parallel training is observed on the production job, in the Custom PySpark action.

In [ ]:
def mape(prediction_col, actual_col="consumption_mw"):
    return F.round(100 * F.avg(F.abs(F.col(prediction_col) - F.col(actual_col)) / F.col(actual_col)), 2)

by_region = (
    predictions.groupBy("region_code", "region_name")
    .agg(
        F.first("train_hours").alias("train_hours"),
        mape("predicted_mw").alias("model_mape_pct"),
        mape("lag_168h").alias("one_week_before_mape_pct"),
    )
    .orderBy("model_mape_pct")
    .toPandas()
)
by_region

The model beats "same hour one week before" in every region, by about a third.
The error ranges from under 4 % (Provence-Alpes-Côte d'Azur, Hauts-de-France) to about 6.5 % (Centre-Val de Loire), with no simple link to the size of the region.

## 6. Add up the regions

The 12 regional forecasts are added hour by hour, on the hours where all 12 regions have a forecast,
and compared with the sum of the actual regional consumption.

In [ ]:
regions_sum = (
    predictions.groupBy("hour_utc")
    .agg(
        F.count("*").alias("regions"),
        F.sum("predicted_mw").alias("predicted_mw"),
        F.sum("consumption_mw").alias("consumption_mw"),
        F.sum("lag_168h").alias("lag_168h"),
    )
    .where(F.col("regions") == 12)
)

print(regions_sum.select(
    F.count("*").alias("hours"),
    mape("predicted_mw").alias("regional_models_mape_pct"),
    mape("lag_168h").alias("one_week_before_mape_pct"),
).first())

Added up, the regional errors partly cancel out: in 2024, the sum of the regional models reaches about 3.5 %,
against about 6 % for the forecast "same hour one week before".
Most of the remaining error is the weather, which this model does not use.

In [ ]:
week = (
    regions_sum
    .withColumn("hour_paris", F.from_utc_timestamp(F.col("hour_utc").cast("timestamp"), "Europe/Paris"))
    .where(F.col("hour_paris").between("2024-01-08 00:00:00", "2024-01-14 23:00:00"))
    .select("hour_paris", "consumption_mw", "predicted_mw")
    .orderBy("hour_paris")
    .toPandas()
    .set_index("hour_paris")
    / 1000
)
week.rename(columns={"consumption_mw": "Actual (sum of regions)", "predicted_mw": "Sum of regional forecasts"}).plot(figsize=(12, 4))
plt.title("Week of 8 January 2024 (Paris time)")
plt.ylabel("GW")
plt.grid(True, alpha=0.3)
plt.show()